# ResNet-18: baseline and AugMix

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/A3_Recv/blob/main/finished/colabs/01_resnet18_augmix.ipynb)

I started with an ImageNet-pretrained ResNet-18 and compared basic augmentation with AugMix. Both runs use the same initial weights, image order and bilinear resize from 32×32 to 128×128.

| Run | Training loss and augmentation | Public Kaggle score |
|---|---|---:|
| Baseline | Crop + flip, cross-entropy | 0.61093 |
| AugMix | Same crop + flip, two AugMix views, cross-entropy + JSD | 0.75796 |

AugMix improved the public score by **14.703 percentage points**. The gap between clean CIFAR validation and synthetic test performance was still large.

I used a maximum of 30 epochs per model, with early stopping after eight epochs without improvement from epoch 15 onward. Each training call has a 2.5-hour budget and can resume from Drive. An A100 runtime is suitable for these runs.


## Notebook version

The next cell links to the latest GitHub revision. Checkpoints remain in Drive when opening a fresh copy.


In [ ]:
import json
import re
import time
import urllib.request
from IPython.display import HTML, display

try:
    request = urllib.request.Request(
        'https://api.github.com/repos/rivas-vega/A3_Recv/commits/main?t=' + str(time.time_ns()),
        headers={'Accept': 'application/vnd.github+json', 'User-Agent': 'A3-Recv-Colab'})
    with urllib.request.urlopen(request, timeout=20) as response:
        latest_sha = json.load(response)['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', latest_sha)
    latest_url = ('https://colab.research.google.com/github/rivas-vega/A3_Recv/blob/'
                  + latest_sha + '/finished/colabs/01_resnet18_augmix.ipynb')
    display(HTML(f'<a href="{latest_url}" target="_blank" rel="noopener noreferrer">'
                 f'Open latest version in Colab ({latest_sha[:7]})</a>'))
except Exception as exc:
    print('Could not reach GitHub; this copy is still usable:', exc)

## 1. Setup and data

`USE_DRIVE=True` keeps checkpoints across Colab sessions. I keep the same `RUN_NAME` to resume a run and use a new name when changing the experiment.

Kaggle authentication uses the `KAGGLE_API_TOKEN` Colab secret, with an interactive login as a fallback. The account must have joined `mva-recvis-2026`. Tokens are available under [Kaggle settings](https://www.kaggle.com/settings/api).


In [ ]:
%pip -q install kagglehub

In [ ]:
from pathlib import Path
import csv
import gc
import hashlib
import json
import math
import os
import random
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms as T
from torchvision.transforms import InterpolationMode
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing.')
DEVICE = torch.device('cuda')
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

USE_DRIVE = True
RUN_NAME = 'resnet18_augmix_128_v1'
HOURS_PER_MODEL = 2.5  # budget per training call; resumable
NUM_WORKERS = min(4, os.cpu_count() or 1)
EVAL_BATCH_SIZE = 256
RESUME = True

CFG = dict(
    code_version='resnet18_augmix_v1', seed=42, image_size=128,
    weights='ResNet18_Weights.IMAGENET1K_V1', batch_size=128,
    max_epochs=30, min_epochs=15, patience=8, warmup_epochs=2,
    backbone_lr=3e-4, head_lr=1e-3, weight_decay=1e-4,
    augmix_severity=3, augmix_width=3, augmix_depth=-1,
    augmix_alpha=1.0, augmix_all_ops=True, jsd_weight=12.0,
    resize='bilinear_antialias', crop_padding=4,
    torch_version=str(torch.__version__), torchvision_version=str(torchvision.__version__),
    amp_dtype=str(AMP_DTYPE),
)
assert 1 <= CFG['min_epochs'] <= CFG['max_epochs']
assert HOURS_PER_MODEL > 0
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_ROOT = Path('/content/drive/MyDrive/A3_Recv/runs') / RUN_NAME
else:
    RUN_ROOT = Path('/content/recvis_runs') / RUN_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0), '| AMP:', AMP_DTYPE)
print('Checkpoints and results:', RUN_ROOT)
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)

In [ ]:
import kagglehub

if not os.environ.get('KAGGLE_API_TOKEN'):
    try:
        from google.colab import userdata
        token = userdata.get('KAGGLE_API_TOKEN')
        if token:
            os.environ['KAGGLE_API_TOKEN'] = token
        del token
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        pass  # Fall back to interactive login below.
kagglehub.login()
download_root = Path(kagglehub.competition_download('mva-recvis-2026'))
data_files = {}
for filename in ('train.npz', 'validation.npz', 'test.npz', 'sample_submission.csv'):
    matches = list(download_root.rglob(filename))
    if len(matches) != 1:
        raise RuntimeError(f'Expected one {filename}; found: {matches}')
    data_files[filename] = matches[0]

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def load_labeled(path):
    with np.load(path, allow_pickle=False) as archive:
        images, labels = archive['images'], archive['labels']
    assert images.dtype == np.uint8 and images.shape[1:] == (32, 32, 3)
    assert np.issubdtype(labels.dtype, np.integer)
    labels = labels.reshape(-1).astype(np.int64)
    assert len(images) == len(labels) and len(labels) > 0
    assert ((labels >= 0) & (labels < 10)).all()
    return images, labels

train_images, train_labels = load_labeled(data_files['train.npz'])
val_images, val_labels = load_labeled(data_files['validation.npz'])
CFG['data_sha256'] = {name: sha256_file(path) for name, path in data_files.items()}
config_path = RUN_ROOT / 'config.json'
if config_path.exists():
    if json.loads(config_path.read_text()) != CFG:
        raise ValueError('Configuration or data changed; use a new RUN_NAME.')
else:
    config_path.write_text(json.dumps(CFG, indent=2))
print('Train:', train_images.shape, '| Validation:', val_images.shape)

## 2. Augmentation and loss

Crop, flip and AugMix operate at 32×32. Each view is then resized once to 128×128 and normalized with ImageNet statistics. Validation and test use only resize and normalization.

The three views share the same basic crop and flip. Per-image, per-epoch seeds keep these transformations consistent when resuming and across the two runs.

The baseline uses cross-entropy. AugMix uses cross-entropy on the basic view plus **12 × JSD** across the three predictions, with JSD computed in float32. I use severity 3, three branches and `all_ops=True`, including color and contrast operations.

References: [torchvision AugMix](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.AugMix.html), [AugMix implementation](https://github.com/google-research/augmix).


In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
basic_transform = T.Compose([
    T.RandomCrop(32, padding=CFG['crop_padding'], padding_mode='reflect'),
    T.RandomHorizontalFlip(),
])
preprocess = T.Compose([
    T.Resize((CFG['image_size'], CFG['image_size']),
             interpolation=InterpolationMode.BILINEAR, antialias=True),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
augmix_transform = T.AugMix(
    severity=CFG['augmix_severity'], mixture_width=CFG['augmix_width'],
    chain_depth=CFG['augmix_depth'], alpha=CFG['augmix_alpha'],
    all_ops=CFG['augmix_all_ops'], interpolation=InterpolationMode.BILINEAR,
)

class TrainDataset(Dataset):
    def __init__(self, images, labels, method, epoch):
        assert method in ('baseline', 'augmix')
        self.images, self.labels = images, labels
        self.method, self.epoch = method, epoch

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = Image.fromarray(self.images[index])
        # Per-image CPU RNG keeps augmentation independent of worker count.
        with torch.random.fork_rng(devices=[]):
            torch.random.default_generator.manual_seed(
                CFG['seed'] + self.epoch * len(self.images) + index)
            image = basic_transform(image)
            clean = preprocess(image)
            if self.method == 'augmix':
                views = (clean, preprocess(augmix_transform(image)),
                         preprocess(augmix_transform(image)))
            else:
                views = clean
        return views, int(self.labels[index])

class EvalDataset(Dataset):
    def __init__(self, images, targets):
        assert len(images) == len(targets)
        self.images, self.targets = images, targets

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        return preprocess(Image.fromarray(self.images[index])), int(self.targets[index])

def make_loader(dataset, training=False, epoch=0):
    generator = torch.Generator().manual_seed(CFG['seed'] + epoch)
    return DataLoader(
        dataset, batch_size=CFG['batch_size'] if training else EVAL_BATCH_SIZE,
        shuffle=training, drop_last=False, num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == 'cuda', generator=generator,
        persistent_workers=False,
    )

def js_divergence(logits):
    log_prob = torch.stack([F.log_softmax(x.float(), dim=1) for x in logits])
    prob = log_prob.exp()
    log_mean = torch.logsumexp(log_prob, dim=0) - math.log(len(logits))
    return (prob * (log_prob - log_mean.unsqueeze(0))).sum(dim=-1).mean()

def classification_loss(logits, targets, method):
    if method == 'baseline':
        clean_logits = logits
        ce = F.cross_entropy(clean_logits.float(), targets)
        jsd = ce.new_zeros(())
    elif method == 'augmix':
        clean_logits, augmented1, augmented2 = logits.split(len(targets))
        ce = F.cross_entropy(clean_logits.float(), targets)
        jsd = js_divergence((clean_logits, augmented1, augmented2))
    else:
        raise ValueError(method)
    return ce + CFG['jsd_weight'] * jsd, ce, jsd, clean_logits

val_loader = make_loader(EvalDataset(val_images, val_labels))

### Check the augmented views

Eight random training images, without changing the training RNG state. This helps me check that the augmentation still leaves the object recognizable.


In [ ]:
indices = np.random.default_rng(CFG['seed']).choice(len(train_images), 8, replace=False)
preview_set = TrainDataset(train_images, train_labels, 'augmix', epoch=1)
fig, axes = plt.subplots(8, 3, figsize=(7, 16))
mean_t = torch.tensor(MEAN)[:, None, None]
std_t = torch.tensor(STD)[:, None, None]
for row, index in enumerate(indices):
    views, _ = preview_set[int(index)]
    for col, (view, title) in enumerate(zip(views, ('Basic', 'AugMix 1', 'AugMix 2'))):
        axes[row, col].imshow((view * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy())
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(title)
plt.tight_layout()
plt.show()

## 3. Training and resuming

Both runs fine-tune all ResNet-18 layers with AdamW: backbone LR `3e-4`, head LR `1e-3`, two warm-up epochs and cosine decay. I select the checkpoint with the highest clean validation accuracy, breaking ties by loss.

`last.pt` stores the model, optimizer, AMP state, history and RNG state after each epoch. An interrupted epoch is repeated. A time-limit pause is resumable; early stopping and the maximum epoch count finish the run. The two models can therefore select different epochs.


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def cpu_state(model):
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

def atomic_save(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    torch.save(value, temporary)
    os.replace(temporary, path)

def write_history(history, path):
    if not history:
        return
    with open(path, 'w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(history[0]))
        writer.writeheader()
        writer.writerows(history)

def build_model(pretrained=False):
    weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model

def lr_factor(epoch):
    warmup, maximum = CFG['warmup_epochs'], CFG['max_epochs']
    if epoch <= warmup:
        return epoch / max(1, warmup)
    progress = (epoch - warmup) / max(1, maximum - warmup)
    return 0.01 + 0.99 * (1 + math.cos(math.pi * progress)) / 2

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    loss_sum = correct = count = 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == 'cuda'):
            logits = model(images)
        loss_sum += F.cross_entropy(logits.float(), labels, reduction='sum').item()
        correct += (logits.argmax(1) == labels).sum().item()
        count += len(labels)
    return loss_sum / count, correct / count

def train_epoch(model, loader, optimizer, scaler, method):
    model.train()
    sums = np.zeros(3, dtype=np.float64)
    correct = count = 0
    for views, labels in loader:
        # Batch order: basic views, first AugMix views, second AugMix views.
        images = torch.cat(views, dim=0) if method == 'augmix' else views
        images = images.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == 'cuda'):
            logits = model(images)
        loss, ce, jsd, clean_logits = classification_loss(logits, labels, method)
        if not torch.isfinite(loss):
            raise FloatingPointError('Non-finite loss; the last complete checkpoint is retained.')
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        size = len(labels)
        sums += np.array([loss.item(), ce.item(), jsd.item()]) * size
        correct += (clean_logits.argmax(1) == labels).sum().item()
        count += size
    return dict(train_loss=sums[0] / count, train_ce=sums[1] / count,
                train_jsd=sums[2] / count, train_accuracy=correct / count)

def train_model(method):
    assert method in ('baseline', 'augmix')
    seed_everything(CFG['seed'])
    run_dir = RUN_ROOT / method
    run_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = run_dir / 'last.pt'
    if checkpoint_path.exists() and not RESUME:
        raise ValueError('Checkpoints exist; use RESUME=True or a new RUN_NAME.')
    model = build_model().to(DEVICE, memory_format=torch.channels_last)
    model.load_state_dict(INITIAL_STATE)
    optimizer = torch.optim.AdamW([
        {'params': [p for name, p in model.named_parameters() if not name.startswith('fc.')],
         'lr': CFG['backbone_lr']},
        {'params': model.fc.parameters(), 'lr': CFG['head_lr']},
    ], weight_decay=CFG['weight_decay'])
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda' and AMP_DTYPE == torch.float16)
    history, best_state = [], None
    best_accuracy, best_loss, best_epoch, bad_epochs = -1.0, float('inf'), 0, 0
    start_epoch, finished = 1, False
    stop_reason = 'max_epochs'
    if checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
        if checkpoint['config'] != CFG or checkpoint['method'] != method:
            raise ValueError('Incompatible checkpoint; use a new RUN_NAME.')
        model.load_state_dict(checkpoint['model'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        scaler.load_state_dict(checkpoint['scaler'])
        history, best_state = checkpoint['history'], checkpoint['best_model']
        best_accuracy, best_loss = checkpoint['best_accuracy'], checkpoint['best_loss']
        best_epoch, bad_epochs = checkpoint['best_epoch'], checkpoint['bad_epochs']
        start_epoch, finished = checkpoint['epoch'] + 1, checkpoint['finished']
        stop_reason = checkpoint['stop_reason']
        torch.set_rng_state(checkpoint['torch_rng'])
        if DEVICE.type == 'cuda' and checkpoint['cuda_rng'] is not None:
            torch.cuda.set_rng_state_all(checkpoint['cuda_rng'])
        print(f'{method}: resumed from epoch {start_epoch - 1}; best validation {best_accuracy:.2%}.')
        del checkpoint

    started = time.monotonic()
    previous_seconds = history[-1]['epoch_seconds'] if history else 0
    if not finished:
        for epoch in range(start_epoch, CFG['max_epochs'] + 1):
            elapsed = time.monotonic() - started
            # Allow at least one epoch per resumed call.
            if epoch > start_epoch and elapsed + 1.15 * previous_seconds > HOURS_PER_MODEL * 3600:
                stop_reason = 'time_budget'
                print(f'{method}: time budget reached; rerun this cell to resume.')
                break
            epoch_started = time.monotonic()
            for group, base_lr in zip(optimizer.param_groups, (CFG['backbone_lr'], CFG['head_lr'])):
                group['lr'] = base_lr * lr_factor(epoch)
            loader = make_loader(TrainDataset(train_images, train_labels, method, epoch),
                                 training=True, epoch=epoch)
            train_metrics = train_epoch(model, loader, optimizer, scaler, method)
            val_loss, val_accuracy = evaluate(model, val_loader)
            improved = (val_accuracy > best_accuracy or
                        (val_accuracy == best_accuracy and val_loss < best_loss))
            if improved:
                best_accuracy, best_loss, best_epoch = val_accuracy, val_loss, epoch
                best_state, bad_epochs = cpu_state(model), 0
            else:
                bad_epochs += 1
            history.append(dict(epoch=epoch, **{k: float(v) for k, v in train_metrics.items()},
                                val_loss=val_loss, val_accuracy=val_accuracy,
                                backbone_lr=optimizer.param_groups[0]['lr'],
                                epoch_seconds=time.monotonic() - epoch_started))
            finished = epoch == CFG['max_epochs'] or (epoch >= CFG['min_epochs'] and bad_epochs >= CFG['patience'])
            stop_reason = 'early_stopping' if finished and epoch < CFG['max_epochs'] else ('max_epochs' if finished else 'running')
            atomic_save(dict(
                config=CFG, method=method, epoch=epoch, model=cpu_state(model),
                optimizer=optimizer.state_dict(), scaler=scaler.state_dict(),
                best_model=best_state, best_accuracy=best_accuracy, best_loss=best_loss,
                best_epoch=best_epoch, bad_epochs=bad_epochs, history=history,
                finished=finished, stop_reason=stop_reason,
                torch_rng=torch.get_rng_state(),
                cuda_rng=torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else None,
            ), checkpoint_path)
            previous_seconds = time.monotonic() - epoch_started
            # CSV timing includes saving; checkpoint timing ends before the save.
            history[-1]['epoch_seconds'] = previous_seconds
            write_history(history, run_dir / 'history.csv')
            print(f"{method} {epoch:02d}/{CFG['max_epochs']} | train {train_metrics['train_accuracy']:.2%}"
                  f' | val {val_accuracy:.2%} | {previous_seconds / 60:.1f} min'
                  + (' | mejor' if improved else ''), flush=True)
            if epoch == start_epoch:
                remaining = (CFG['max_epochs'] - epoch) * previous_seconds / 3600
                print(f'Estimated time for remaining epochs of {method}: {remaining:.2f} h; depends on Colab throughput.')
            del loader
            if finished:
                break
    if best_state is None:
        raise RuntimeError('No valid trained checkpoint is available for export.')
    model.load_state_dict(best_state)
    model.eval()
    # Rebuild the CSV if the session ended immediately after saving last.pt.
    write_history(history, run_dir / 'history.csv')
    atomic_save({'model': best_state, 'config': CFG, 'method': method,
                 'epoch': best_epoch, 'val_accuracy': best_accuracy}, run_dir / 'best.pt')
    summary = dict(method=method, epochs_trained=len(history), best_epoch=best_epoch,
                   val_accuracy=best_accuracy, val_loss=best_loss,
                   training_hours=sum(row['epoch_seconds'] for row in history) / 3600,
                   stop_reason=stop_reason, completed=finished)
    (run_dir / 'summary.json').write_text(json.dumps(summary, indent=2))
    print(f'{method}: using epoch {best_epoch}, validation {best_accuracy:.2%}.')
    del optimizer, scaler, best_state
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return model, summary

In [ ]:
seed_everything(CFG['seed'])
initial_path = RUN_ROOT / 'initial_state.pt'
if initial_path.exists():
    initial_checkpoint = torch.load(initial_path, map_location='cpu', weights_only=True)
    if initial_checkpoint['config'] != CFG:
        raise ValueError('Initialization mismatch; use a new RUN_NAME.')
    INITIAL_STATE = initial_checkpoint['model']
    del initial_checkpoint
else:
    initial_model = build_model(pretrained=True)
    INITIAL_STATE = cpu_state(initial_model)
    atomic_save({'model': INITIAL_STATE, 'config': CFG}, initial_path)
    del initial_model
print('Shared ImageNet initialization ready; all layers will be trained.')

## 4. Export both submissions

Inference uses each model's selected checkpoint and the same 128×128 bilinear preprocessing as validation. There is no AugMix or test-time augmentation here.

Predictions are aligned by ID with `sample_submission.csv`. The baseline CSV is saved before starting AugMix so it remains available if the second run is interrupted.


In [ ]:
def write_submission(ids, predictions, template_path, output_path):
    ids = [int(value) for value in ids]
    raw_predictions = np.asarray(predictions)
    if not np.issubdtype(raw_predictions.dtype, np.integer):
        raise ValueError('Predicted labels must be integers.')
    predictions = [int(value) for value in raw_predictions]
    if len(ids) != len(predictions) or len(set(ids)) != len(ids):
        raise ValueError('Duplicate IDs or incorrect length.')
    if not predictions or not all(0 <= value <= 9 for value in predictions):
        raise ValueError('Predictions are empty or outside [0,9].')
    with open(template_path, newline='') as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames != ['id', 'label']:
            raise ValueError('The submission template must have columns id,label.')
        ordered_ids = [int(row['id']) for row in reader]
    if len(ordered_ids) != len(ids) or len(set(ordered_ids)) != len(ordered_ids) or set(ordered_ids) != set(ids):
        raise ValueError('Test IDs do not match the submission template.')
    lookup = dict(zip(ids, predictions))
    output_path = Path(output_path)
    temporary = output_path.with_name(output_path.name + '.tmp')
    with open(temporary, 'w', newline='') as handle:
        writer = csv.writer(handle)
        writer.writerow(['id', 'label'])
        writer.writerows((identifier, lookup[identifier]) for identifier in ordered_ids)
    os.replace(temporary, output_path)
    return len(ordered_ids)

@torch.inference_mode()
def make_submission(model, method):
    with np.load(data_files['test.npz'], allow_pickle=False) as archive:
        test_images, test_ids = archive['images'], archive['ids']
    assert test_images.dtype == np.uint8 and test_images.shape[1:] == (32, 32, 3)
    assert test_ids.ndim == 1 and np.issubdtype(test_ids.dtype, np.integer)
    test_loader = make_loader(EvalDataset(test_images, test_ids))
    model.eval()
    ids, predictions = [], []
    for images, batch_ids in test_loader:
        images = images.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == 'cuda'):
            logits = model(images)
        predictions.extend(logits.argmax(1).cpu().tolist())
        ids.extend(batch_ids.tolist())
    output_path = RUN_ROOT / f'submission_{method}.csv'
    count = write_submission(ids, predictions, data_files['sample_submission.csv'], output_path)
    print(f'{output_path.name}: {count:,} checked predictions. Saved to {output_path}')
    return output_path

In [ ]:
baseline_model, baseline_summary = train_model('baseline')
baseline_csv = make_submission(baseline_model, 'baseline')
del baseline_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
augmix_model, augmix_summary = train_model('augmix')
augmix_csv = make_submission(augmix_model, 'augmix')
del augmix_model
gc.collect()
torch.cuda.empty_cache()

## 5. Compare the runs

The table and curves below are loaded from the current run. `KAGGLE_SCORES` is a separate record for manually entered public scores; local validation remains separate from Kaggle evaluation.

If training stopped at the time limit, resume it before treating the comparison as final. Prediction disagreement is useful to inspect, but it does not tell me which model is correct.


In [ ]:
# Public scores for this run, entered after submitting the CSV files.
KAGGLE_SCORES = {'baseline': None, 'augmix': None}
rows = []
for method in ('baseline', 'augmix'):
    summary = json.loads((RUN_ROOT / method / 'summary.json').read_text())
    summary['kaggle_accuracy'] = KAGGLE_SCORES[method]
    summary['csv'] = f'submission_{method}.csv'
    rows.append(summary)
comparison = pd.DataFrame(rows)
comparison.to_csv(RUN_ROOT / 'comparison.csv', index=False)
display(comparison.style.format({'val_accuracy': '{:.2%}', 'val_loss': '{:.4f}',
                                'training_hours': '{:.2f}', 'kaggle_accuracy': '{:.2%}'}, na_rep='pending'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for method in ('baseline', 'augmix'):
    history = pd.read_csv(RUN_ROOT / method / 'history.csv')
    axes[0].plot(history.epoch, 100 * history.val_accuracy, label=method)
    axes[1].plot(history.epoch, history.val_loss, label=method)
axes[0].set(xlabel='Epoch', ylabel='Clean validation accuracy (%)')
axes[1].set(xlabel='Epoch', ylabel='Validation cross-entropy')
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(RUN_ROOT / 'validation_curves.png', dpi=150)
plt.show()

baseline_predictions = pd.read_csv(RUN_ROOT / 'submission_baseline.csv').set_index('id')
augmix_predictions = pd.read_csv(RUN_ROOT / 'submission_augmix.csv').set_index('id')
augmix_predictions = augmix_predictions.reindex(baseline_predictions.index)
disagreement = float((baseline_predictions.label != augmix_predictions.label).mean())
print(f'Prediction disagreement: {disagreement:.2%} (not accuracy).')
delta = comparison.set_index('method').val_accuracy['augmix'] - comparison.set_index('method').val_accuracy['baseline']
print(f'Validation difference, AugMix − baseline: {100 * delta:+.2f} percentage points.')
if all(KAGGLE_SCORES[m] is not None for m in ('baseline', 'augmix')):
    print(f"Kaggle difference: {100 * (KAGGLE_SCORES['augmix'] - KAGGLE_SCORES['baseline']):+.2f} percentage points.")

In [ ]:
# Bundle both CSV files, the summary and curves.
# Each submission CSV is uploaded separately on Kaggle.
bundle = RUN_ROOT / 'recvis_two_submissions.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for filename in ('submission_baseline.csv', 'submission_augmix.csv', 'comparison.csv',
                     'validation_curves.png', 'config.json'):
        archive.write(RUN_ROOT / filename, arcname=filename)
    for method in ('baseline', 'augmix'):
        archive.write(RUN_ROOT / method / 'history.csv', arcname=f'{method}_history.csv')
print('Also saved in Drive:', RUN_ROOT)
from google.colab import files
files.download(str(bundle))

## Notes

This comparison keeps the architecture, ImageNet initialization and input resolution fixed while changing augmentation and the consistency loss. The large Kaggle improvement motivated trying a stronger pretrained representation next.

The download above contains predictions and run summaries. The course code archive is separate and excludes checkpoints and data.
